# Classificador de Desempenho de Atletas

A ideia do projeto é utilizar inteligencia artificial para medir o desempenho de um atleta em uma partida com base nos seus dados. O modelo classifica entre "Baixo Desempenho", "Médio Desempenho" e "Alto Desempenho" utilizando os outros jogadores da mesma função. O objetivo do projeto é ajudar treinadores a avaliarem melhor seus atletas, gerando um bom comparativo com jogadores da mesma categoria.

O modelo é uma rede neural de classificação, que foi treinada com dados de partidas de atletas podendo ser acessadas pelo arquivo de "AmostraDados-ABP-5DSM-Grande", uma amostra expandida dos dados originais, com mais de 38 mil amostras.

O código é divido entre:

- **Configurações**: onde se coloca os parametros utilizados.
- **Carregamento de Dados**: uma parte somente para carregar e configurar os dados para o treino.
- **Normalização**: os dados são normalizados para o melhor treino da IA.
- **Geração de Rótulos**: como os dados não possuem uma resposta direta, eles são rotulados para o treinamento.
- **Treinamento**: com os dados já formatados e as as respostas prontas, se começa o treinamento do modelo.

<br />

---



## ⚙️ Configurações

Aqui foi colocado os caminhos dos arquivos utilizados e o nome das classificações. É importante a criação de arquivos externos para utilização em outros projetos.

Também aqui é possivel alterar os pesos iniciais de cada dado, é importante que algumas colunas tenham mais peso que outras inicialmente, já que há medições que afetam mais a qualidade do atleta do que outras.

In [15]:
CAMINHO_CSV         = "../data/AmostraDados-ABP-5DSM-Grande.csv"
CAMINHO_MODELO      = "../model/modelo_atletas.onnx"
CAMINHO_NORMALIZADOR = "../model/normalizador_atletas.onnx"
NOME_CLASSES = {0: "Baixo Desempenho", 1: "Médio Desempenho", 2: "Alto Desempenho"}

PESOS_FEATURES = {
    "posicao_numerica"            : 0.0,
    "Workload"                    : 1.5,
    "Sprint Distance"             : 1.5,
    "High Intensity Running"      : 1.5,
    "Top Speed"                   : 1.0,
    "Accelerations"               : 1.0,
    "Decelerations"               : 1.0,
    "No. of Sprints"              : 1.2,
    "Metres per Minute"           : 1.5,
    "No. of High Intensity Events": 1.0,
    "Minutes Played"              : 0.8,
}

## 🗃️ Carregamento dos Dados

Nesta parte os dados são carregados e colocados em um tabela por meio da biblioteca `pandas`. Também são formatados, onde atletas que tem grupos vazios, ficam como "Indefinido".

Também é criado um mapa de posições, que basicamente transforma as posições em números para que o modelo consiga entender. Junto com isso todos os dados são colocados como tipo `float`.

In [16]:
import pandas as pd

tabela = pd.read_csv(CAMINHO_CSV)

tabela["Group"] = tabela["Group"].fillna("Indefinido")

mapa_posicoes = {pos: i for i, pos in enumerate(["10s", "CBs", "CMs", "STs", "WBs", "Indefinido"])}
tabela["posicao_numerica"] = tabela["Group"].map(mapa_posicoes)

colunas_features = list(PESOS_FEATURES.keys())

dados = tabela[colunas_features].values.astype(float)

## 🔁 Normalização

Para evitar que o modelo classificasse valores altos como um peso muito grande, e valores baixos como pesos baixos, há uma normalização desses dados, onde todos ficam na faixa de 0 a 1.

O normalizador criado da base de dados é salvo para que possa ser utilizado futuramente para fazer predições.

In [17]:
from sklearn.preprocessing import MinMaxScaler
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

normalizador = MinMaxScaler()
dados_normalizados = normalizador.fit_transform(dados)

initial_type = [('input', FloatTensorType([None, dados.shape[1]]))]
onnx_normalizer = convert_sklearn(normalizador, initial_types=initial_type)

with open(CAMINHO_NORMALIZADOR, "wb") as f:
    f.write(onnx_normalizer.SerializeToString())    # type: ignore

print(f"Normalizador salvo em '{CAMINHO_NORMALIZADOR}'")

Normalizador salvo em '../model/normalizador_atletas.onnx'


## 🏷️ Gerar Rótulos

Como os dados não possuem uma resposta, é necessario criá-las, o metodo escolhido foi rankear os atletas e gerar uma divisão em porcentagem para a classificação com base no seu grupo, então zagueiros são classificados com base em outros zagueiros, atacantes com atacantes, já que a métrica de desempenho dessas funções podem ter uma leve mudança.

In [18]:
import numpy as np

vetor_pesos = np.array(list(PESOS_FEATURES.values()))
escores     = (dados_normalizados * vetor_pesos).sum(axis=1) / vetor_pesos.sum()

tabela_com_escores = tabela.copy()
tabela_com_escores['escores'] = escores

tabela_com_escores['rotulos_grupo'] = -1

for group_name in tabela_com_escores['Group'].unique():
    group_mask = (tabela_com_escores['Group'] == group_name)
    escores_grupo = tabela_com_escores.loc[group_mask, 'escores']

    corte_baixo_grupo = np.percentile(escores_grupo, 30)    # type: ignore
    corte_alto_grupo  = np.percentile(escores_grupo, 65)    # type: ignore

    rotulos_grupo_calculado = np.where(escores_grupo <= corte_baixo_grupo, 0,
                              np.where(escores_grupo <= corte_alto_grupo,  1, 2))

    tabela_com_escores.loc[group_mask, 'rotulos_grupo'] = rotulos_grupo_calculado

rotulos = tabela_com_escores['rotulos_grupo'].values

print("\nDistribuição de classes:")
for classe, nome in NOME_CLASSES.items():
    print(f"  {nome}: {(rotulos == classe).sum()} atletas") # type: ignore


Distribuição de classes:
  Baixo Desempenho: 11526 atletas
  Médio Desempenho: 13443 atletas
  Alto Desempenho: 13447 atletas


## 🤖 Treinamento

Primeiramente é dividido os dados em dados para treino e dados para teste, na razão de 80% e 20%. O modelo escolhido foi uma rede neural com 2 camadas ocultas com 32 e 16 neurônios, com até 1500 épocas, onde em cada uma ele faz os seguintes passos:

1. faz forward propagation
2. calcula o erro
3. aplica backpropagation
4. atualiza os pesos

Depois do treinamento ele é salvo em um arquivo para utilização.

In [19]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

x_train, x_test, y_train, y_test = train_test_split(dados_normalizados, rotulos, test_size=0.2, random_state=67)

modelo = MLPClassifier(
    hidden_layer_sizes = (32, 16),  # arquitetura da rede
    activation         = "relu",    # função de ativação
    max_iter           = 1500,      # número de épocas
    learning_rate_init = 0.005,     # taxa de aprendizado
    random_state       = 67,        # reprodutibilidade
    verbose            = True,
)

print("\nTreinando a rede neural...")
modelo.fit(x_train, y_train)

acuracia_treino = modelo.score(x_train, y_train)
acuracia_teste = modelo.score(x_test, y_test)
print(f"Acurácia no treino: {acuracia_treino * 100:.1f}%")
print(f"Acurácia no teste: {acuracia_teste * 100:.1f}%")

initial_type_model = [('input', FloatTensorType([None, x_train.shape[1]]))]
onnx_model = convert_sklearn(
    modelo,
    initial_types=initial_type_model,
    target_opset=13,
    options={id(modelo): {"zipmap": False}}
)

with open(CAMINHO_MODELO, "wb") as f:
    f.write(onnx_model.SerializeToString()) # type: ignore

print(f"Modelo salvo em '{CAMINHO_MODELO}'")


Treinando a rede neural...
Iteration 1, loss = 0.41843836
Iteration 2, loss = 0.24219312
Iteration 3, loss = 0.22796746
Iteration 4, loss = 0.22029222
Iteration 5, loss = 0.21331748
Iteration 6, loss = 0.20704315
Iteration 7, loss = 0.20647169
Iteration 8, loss = 0.20164738
Iteration 9, loss = 0.19720258
Iteration 10, loss = 0.19403949
Iteration 11, loss = 0.19000300
Iteration 12, loss = 0.18313544
Iteration 13, loss = 0.18150924
Iteration 14, loss = 0.17901341
Iteration 15, loss = 0.17478801
Iteration 16, loss = 0.17261498
Iteration 17, loss = 0.16711628
Iteration 18, loss = 0.16489966
Iteration 19, loss = 0.16001957
Iteration 20, loss = 0.14950900
Iteration 21, loss = 0.14713280
Iteration 22, loss = 0.13563462
Iteration 23, loss = 0.13051761
Iteration 24, loss = 0.12931616
Iteration 25, loss = 0.11955963
Iteration 26, loss = 0.12023825
Iteration 27, loss = 0.11386665
Iteration 28, loss = 0.10601517
Iteration 29, loss = 0.10606132
Iteration 30, loss = 0.09941759
Iteration 31, loss = 